# r9p step C on Colab or any CUDA machine: train r9p from r8p
Same data and training as `C_train` (Kaggle). First put `best.pt` (v10-r8p) and `real_questions_full.tgz` in `MyDrive/kinthread_r9p` (or set `KIN_R9P`). Checkpoints and results are written there; if the session drops, run all cells again and training resumes. About 3-6 h on a T4.

In [ ]:
import os, glob, json, hashlib, shutil, subprocess
# Put two private files in IN (never in git):
#   best.pt                  v10-r8p, SHA-256 38c289d3... (Kaggle notebook99ae0586bd output, or best.zip on the Mac)
#   real_questions_full.tgz  the file in the private Kaggle dataset kinthread-real-questions
IN = os.environ.get('KIN_R9P', '/content/drive/MyDrive/kinthread_r9p')
if IN.startswith('/content/drive/') and not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
W = IN
PUB, PRIV = f'{W}/public', f'{W}/private'   # public: counts only, safe to commit; private: real question text, never commit
os.makedirs(PUB, exist_ok=True)
os.makedirs(PRIV, exist_ok=True)
TMP = os.environ.get('KIN_TMP', '/content/temp' if os.path.isdir('/content') else '/tmp/kinthread-r9p')
REPO, BRANCH = f'{TMP}/Kinthread-llm-model', 'main'
R8P_SHA = '38c289d351d6517b5cde1bc9fb64ea97bd33eee193639d6351edda52fdfd7a52'


def sh(cmd):
    print('$', cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f'STOP: command failed: {cmd}')


def stop(msg):
    raise SystemExit('STOP: ' + msg)


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


import torch
if not torch.cuda.is_available():
    stop('no GPU (Colab: Runtime > Change runtime type > T4 GPU)')
for name in ('best.pt', 'real_questions_full.tgz'):
    if not os.path.exists(f'{IN}/{name}'):
        stop(f'put {name} in {IN}')
os.makedirs(TMP, exist_ok=True)
if not os.path.exists(REPO):
    sh(f'git clone --depth 1 -b {BRANCH} https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}')
os.chdir(REPO)
sh(f'rm -rf data/real_questions && tar -xzf {IN}/real_questions_full.tgz')
if not glob.glob('data/real_questions/*.jsonl'):
    stop('no real question files after unpacking real_questions_full.tgz')
os.makedirs('runs/v10-r8p-final', exist_ok=True)
if not os.path.exists('runs/v10-r8p-final/best.pt'):
    if sha(f'{IN}/best.pt') != R8P_SHA:
        stop(f'{IN}/best.pt is not v10-r8p (SHA-256 {R8P_SHA})')
    shutil.copy(f'{IN}/best.pt', 'runs/v10-r8p-final/best.pt')
print('repo, real questions and r8p checkpoint ready on', torch.cuda.get_device_name(0))

sh('pip -q install -U gdown')
S = f'{TMP}/share'
os.makedirs(S, exist_ok=True)
if not os.path.isdir(f'{S}/dataset_v7'):
    sh(f'gdown -q --folder https://drive.google.com/drive/folders/1MNiQuramSX5ImytxEdvNEIN7SgFtT6k8 -O {S}/dataset_v7')
if not os.path.exists(f'{S}/star_r3.jsonl.gz'):
    sh(f'wget -q -O {S}/star_r3.jsonl.gz "https://drive.usercontent.google.com/download?id=1_qk0JL_lkzKBdv2vwHxWijI4yNrNPRtr&export=download&confirm=t"')
DATA, OUT = f'{S}/dataset_v7', f'{W}/runs/v10-r9p'
os.makedirs(OUT, exist_ok=True)
sh('cd data && sha256sum -c SHA256SUMS --ignore-missing --quiet')

In [ ]:
EXTRA = [f'{DATA}/reworded.jsonl.gz*2', 'data/mix_r8.jsonl.gz', f'{S}/star_r3.jsonl.gz*2',
         'data/balanced_mix_v1.jsonl.gz*2', 'data/detour_v1.jsonl.gz']
missing = [e for e in EXTRA if not os.path.exists(e.split('*')[0])]
if missing:
    stop(f'training files missing: {missing}')
sh('PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_reader.py --arch v10 --run v10-r9p '
   f'--out {OUT} --pool {DATA} --init runs/v10-r8p-final/best.pt --prefix_lm --of_form --style_p 0.8 '
   '--steps 15000 --batch 64 --lr 3e-4 --warmup 500 --optim muon --amp --workers 2 --resume '
   '--eval_every 1500 --save_every 1000 --reworded ' + ','.join(EXTRA) + ' --reworded_dev data/reworded_dev.jsonl.gz '
   '--p_reworded 0.75 --ramp_steps 2000 --block 512 --real_dev 575 --select_real')

In [ ]:
sh(f'rm -f {OUT}/latest.pt')
best = f'{OUT}/best.pt'
if not os.path.exists(best):
    stop('training wrote no best.pt')
sh(f'python -m eval.run_real --ckpt {best} --split dev --adaptive 1.0 --hybrid --out {PRIV}/r9p_dev_quick.json')
quick = json.load(open(f'{PRIV}/r9p_dev_quick.json'))['score']['splits']
summary = {'step': 'C', 'r9p_sha256': sha(best), 'r9p_dev_quick_adaptive_hybrid': quick,
           'r8p_dev_same_setting': 'see colab/r8p_kaggle.ipynb output (dev, --adaptive 1.0 --hybrid)'}
json.dump(summary, open(f'{PUB}/C_summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))
print('C DONE')